# Nokia BTS Finder - Step-by-Step Worked Example

This notebook provides a self-contained, reproducible walkthrough of the **Nokia BTS Finder** localization pipeline.
We reverse-engineer physical cellular base station coordinates (eNodeB) from empirical drive-test smartphone measurements without telecom operator internal topology.

### Pipeline Stages:
1. **Universal Ingestion**: Parse mobile drive-test logs (supports raw Craxiom Network Survey exports).
2. **Spatial Preprocessing**: Metric UTM projection, linear power domain averaging, and spatial grid deduplication.
3. **Empirical Propagation Modeling**: Robust IRLS fitting of the Log-Distance Path Loss model.
4. **Multi-start Huber Multilateration**: Estimating tower coordinates with physical Timing Advance boundary constraints.
5. **Sector Geometry**: Convex Hull coverage polygons and boresight azimuth angle estimation.
6. **Cartographic Visualization**: Publication plots and interactive Folium GIS map.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure package root is in sys.path
module_path = os.path.abspath(".")
if module_path not in sys.path:
    sys.path.append(module_path)

from src.loader import load_measurements, get_dataset_summary
from src.preprocessing import SpatialProjector, spatial_deduplicate
from src.path_loss import PathLossModel
from src.solver import BTSLocalizationSolver
from src.sectors import generate_cell_sectors
from src.visualization import (
    plot_localization_triangulation,
    plot_path_loss_model,
    plot_sector_coverage,
    build_interactive_map
)

print("All modules loaded successfully.")

## 1. Data Ingestion
Load the sample dataset containing real drive-test measurements in Timișoara across 9 target eNodeBs.

In [ ]:
sample_path = os.path.join("data", "sample_lte_measurements.csv")
df_raw = load_measurements(sample_path)
print(f"Loaded {len(df_raw)} records.")

summary = get_dataset_summary(df_raw)
pd.DataFrame(summary)

## 2. Preprocessing & Spatial Deduplication
Select a target eNodeB (e.g. `80609` - Ericsson Orange Macro or `101811` - Nokia Digi Macro), project to UTM metric coordinates, and deduplicate spatially to eliminate stationary bias.

In [ ]:
target_eid = 80609
enb_df = df_raw[df_raw["enodeb_id"] == target_eid].copy()

projector = SpatialProjector(median_lon=float(enb_df["longitude"].median()))
enb_dedup = spatial_deduplicate(enb_df, grid_size_m=10.0, projector=projector)

print(f"Raw samples: {len(enb_df)} -> Deduplicated grid cells: {len(enb_dedup)}")
enb_dedup.head()

## 3. Robust Multilateration Solver
Execute the Huber loss multilateration solver with multi-start seed initialization and physical TA boundary penalties.

In [ ]:
solver = BTSLocalizationSolver(projector=projector)
sol = solver.solve(enb_dedup, eNodeB_id=target_eid)

print("--- LOCALIZATION RESULTS ---")
print(f"Estimated Latitude  : {sol['lat']:.6f}°")
print(f"Estimated Longitude : {sol['lon']:.6f}°")
print(f"Confidence Grade    : {sol['confidence_grade']} (Score: {sol['confidence_score']:.2f})")
print(f"Solver RMSE         : {sol['rmse_m']:.1f} meters")
print(f"Method Used         : {sol['method_used']}")
print(f"Path Loss Exponent n: {sol['path_loss_params']['path_loss_exponent_n']:.2f}")

## 4. Sector Coverage & Azimuth Estimation
Group measurements into cell sectors and compute Convex Hull coverage boundaries.

In [ ]:
sectors = generate_cell_sectors(enb_dedup, tower_lat=sol["lat"], tower_lon=sol["lon"])
for s in sectors:
    print(f"Sector PCI {s['pci']}: {s['point_count']} points, Mean RSRP: {s['rsrp_mean']} dBm, Azimuth: {s.get('estimated_azimuth_deg')}°")

## 5. Visualizations
Generate publication-quality diagnostic charts.

In [ ]:
os.makedirs("output", exist_ok=True)
fig1_path = os.path.join("output", f"fig1_bts_localization_{target_eid}.png")
plot_localization_triangulation(enb_dedup, sol, fig1_path, sectors=sectors)

fig2_path = os.path.join("output", f"fig2_path_loss_model_{target_eid}.png")
plot_path_loss_model(enb_dedup, solver.path_loss, sol["lat"], sol["lon"], fig2_path, enodeb_id=target_eid)

fig3_path = os.path.join("output", f"fig3_sector_coverage_{target_eid}.png")
plot_sector_coverage(enb_dedup, sectors, sol["lat"], sol["lon"], fig3_path, enodeb_id=target_eid)

print("Figures generated in output/ folder.")

## 6. Interactive GIS Map
Build a standalone Folium web map to explore the drive test, sectors, and tower interactively.

In [ ]:
html_path = os.path.join("output", "interactive_map.html")
build_interactive_map([sol], {target_eid: sectors}, enb_df, html_path)
print(f"Interactive map saved to: {html_path}")